# 📚 Research Paper Answer Bot: Grounded RAG over Seminal AI Literature
### Capstone Project | GenAI Pinnacle Plus Program
**Tech Stack:** Python, LangChain, Google Gemini, ChromaDB, Sentence-Transformers, Streamlit

---
## 📌 Project Overview
This notebook implements a complete, end-to-end **Retrieval-Augmented Generation (RAG)** pipeline designed to answer complex scientific and technical questions over a corpus of seminal AI research papers.

### Key Highlights:
1. **Zero Hallucination Guarantee:** Grounded prompt templates with mandatory refusal rules (`'I don't know based on the provided papers.'`).
2. **Strict Source Citations:** Every generated answer is paired with the **Top 3 supporting passages**, complete with exact **Paper Title** and **Page Number** citations.
3. **Multi-Model Embedding Benchmarks:** Quantitative comparison of `all-MiniLM-L6-v2` vs `BAAI/bge-small-en-v1.5`.
4. **Multi-Strategy Retrieval:** Dense Cosine Baseline, Maximal Marginal Relevance (MMR), BM25 Lexical, Hybrid (RRF), and Cross-Encoder Reranking.

In [ ]:
# 1. Environment Setup & Configuration
import os
import sys
from pathlib import Path
import pandas as pd

# Add project root to sys.path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import config
from src.document_loader import load_all_papers
from src.chunking import ChunkingEngine
from src.embeddings import get_embedding_model, benchmark_embedding_models
from src.vector_store import get_vector_store
from src.retrieval import RetrievalEngine
from src.rag_chain import ResearchPaperRAGChain
from src.evaluation import BENCHMARK_QUESTIONS, evaluate_retrieval_strategies

print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Data Directory: {config.DATA_DIR}")
print(f"ChromaDB Persist: {config.CHROMA_PERSIST_DIR}")

---
## 📄 Step 1: Document Loading & Metadata Preservation
Loading all 10 seminal AI research papers and preserving document metadata (`paper_title`, `source_file`, `page_number`, `total_pages`).

In [ ]:
documents = load_all_papers(config.DATA_DIR)
print(f"Successfully loaded {len(documents)} page documents.")

# Display metadata summary of loaded documents
doc_df = pd.DataFrame([
    {
        'Paper Title': doc.metadata.get('paper_title'),
        'File Name': doc.metadata.get('source_file'),
        'Page': doc.metadata.get('page_number'),
        'Char Count': len(doc.page_content)
    }
    for doc in documents
])
doc_df.head(10)

---
## ✂️ Step 2: Semantic Chunking
Splitting document pages into structured text chunks using `RecursiveCharacterTextSplitter` (600 characters with 100 character overlap) to preserve formulaic and theorem context.

In [ ]:
chunker = ChunkingEngine(chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP)
chunks = chunker.split_documents(documents)
print(f"Generated {len(chunks)} chunks across {len(documents)} pages.")

# Inspect a sample chunk with enriched metadata
sample = chunks[0]
print(f"Sample Chunk ID: {sample.metadata.get('chunk_id')}")
print(f"Source: {sample.metadata.get('paper_title')} (Page {sample.metadata.get('page_number')})")
print(f"Content:\n{sample.page_content[:300]}...")

---
## 🔬 Step 3: Embedding Models Benchmark
Evaluating **Model A (`all-MiniLM-L6-v2`)** vs. **Model B (`BAAI/bge-small-en-v1.5`)** on encoding throughput and semantic separation.

In [ ]:
emb_csv = config.EXPERIMENTS_DIR / 'embedding_comparison.csv'
if emb_csv.exists():
    emb_df = pd.read_csv(emb_csv)
    display(emb_df)
else:
    print("Running embedding benchmark...")
    emb_results = benchmark_embedding_models([c.page_content for c in chunks[:40]])
    emb_df = pd.DataFrame(emb_results)
    display(emb_df)

---
## ⚡ Step 4: Vector Store & Retrieval Strategy Benchmark
Benchmarking 5 distinct retrieval strategies (**Dense**, **MMR**, **BM25**, **Hybrid RRF**, and **Cross-Encoder Rerank**) across our 12 ground-truth benchmark questions.

In [ ]:
ret_csv = config.EXPERIMENTS_DIR / 'retrieval_comparison.csv'
if ret_csv.exists():
    ret_df = pd.read_csv(ret_csv)
    display(ret_df)
else:
    print("Evaluating retrieval strategies...")
    ret_df = evaluate_retrieval_strategies()
    display(ret_df)

---
## 🤖 Step 5: End-to-End RAG Q&A with Google Gemini
Executing live Q&A queries through the hybrid RAG chain and verifying zero-hallucination source attribution.

In [ ]:
# Initialize RAG Pipeline
rag_chain = ResearchPaperRAGChain(
    embedding_model_name=config.DEFAULT_EMBEDDING_MODEL,
    retrieval_strategy='hybrid'
)

# Test Query 1: Architectural / Factual
q1 = "What is the computational complexity of self-attention per layer?"
res1 = rag_chain.query(q1, strategy='hybrid', top_k=3)

print(f"QUESTION: {q1}")
print(f"\nAI ANSWER:\n{res1['answer']}")
print(f"\nTOP 3 SUPPORTING SOURCES:")
for i, src in enumerate(res1['sources'], 1):
    print(f"[{i}] {src['paper_title']} | Page {src['page_number']} | Score: {src['score']:.4f}")
    print(f"    Passage: {src['passage'][:180]}...\n")

In [ ]:
# Test Query 2: Negative Control (Hallucination Guardrail Check)
q2 = "What is the clinical trial efficacy of mRNA-1273 in pediatric populations?"
res2 = rag_chain.query(q2, strategy='hybrid', top_k=3)

print(f"QUESTION: {q2}")
print(f"\nAI ANSWER:\n{res2['answer']}")
print(f"Refusal Triggered: {res2['is_refusal']}")

---
## 📊 Step 6: Visualizations & Evaluation Summary
Displaying the generated EDA and evaluation figures.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_files = [
    ('Fig 1: Paper Page Distribution', config.VISUALIZATIONS_DIR / 'eda_1_paper_page_distribution.png'),
    ('Fig 2: Chunk Length Distribution', config.VISUALIZATIONS_DIR / 'eda_2_chunk_length_distribution.png'),
    ('Fig 4: Retrieval Strategy Benchmark', config.VISUALIZATIONS_DIR / 'eda_4_retrieval_strategy_comparison.png'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, img_path) in zip(axes, fig_files):
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')
plt.tight_layout()
plt.show()